In [1]:
tabla='qtsop10'
col_tabla = "solopefec"

In [2]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time 
from datetime import datetime
from sqlalchemy import text
import oracledb
import sys
import psycopg2

In [3]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [4]:
# fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=13", con=connection2)

# fecha= fecha.iloc[0, 0]
# print(fecha)
# now = datetime.now()
# query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=13"
# c1= text(query)
# connection2.execute(c1)

In [9]:
fecha = '2023-09-21'

In [10]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

base2 = pd.read_sql_query(f"SELECT * FROM {tabla} WHERE {col_tabla} >= TO_DATE('{fecha}', 'YYYY-MM-DD')", con=connection0)



connection0.close()

In [11]:
base2.shape

(2743, 63)

In [7]:
base2.head()

,solopeoricenasicod,solopecenasicod,solopenum,solopefec,solopeactmednum,solopemedtipdocidenpercod,solopemedperasisdocidennum,solopeinfmed,solopesolfec,solopeprofec,...,soloperiequifec,soloperieneufec,solopeevalpqxmedtipdoc,solopeevalpqxmeddocnum,solopediashospreflg,solopediashosposflg,solopereqprotflg,solopereqprotdes,solopetieprotflg,solopetlffamnum
0,1,001,115341,2023-09-01,13600794,1,29312158,DOLOR PERSISTENTE POR FRACTURA L1,2023-09-02 00:00:00,2023-09-02 00:00:00,...,None,None,None,None,1,1,1,KIT DE VERTEBROPLASTIA,0,999770533
1,1,001,124774,2023-09-01,14067548,1,07244049,Ok,2023-09-02 00:00:00,2023-09-02 00:00:00,...,None,None,None,None,None,None,None,None,None,None
2,1,001,124885,2023-09-01,14099256,1,29534641,Ok,2023-09-02 00:00:00,2023-09-02 00:00:00,...,None,None,None,None,None,None,None,None,None,None
3,1,001,125575,2023-09-01,14067727,1,08184793,Ok,2023-09-02 00:00:00,2023-09-02 00:00:00,...,None,None,None,None,None,None,None,None,None,None
4,1,001,125584,2023-09-01,14119450,1,08011305,Ok,2023-09-01 00:00:00,2023-09-01 00:00:00,...,None,None,None,None,None,None,None,None,None,None


In [20]:
print(base2.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65083 entries, 0 to 65082
Data columns (total 63 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   solopeoricenasicod          65083 non-null  object        
 1   solopecenasicod             65083 non-null  object        
 2   solopenum                   65083 non-null  int64         
 3   solopefec                   65083 non-null  datetime64[ns]
 4   solopeactmednum             65083 non-null  int64         
 5   solopemedtipdocidenpercod   65083 non-null  object        
 6   solopemedperasisdocidennum  65083 non-null  object        
 7   solopeinfmed                65083 non-null  object        
 8   solopesolfec                65083 non-null  object        
 9   solopeprofec                40025 non-null  object        
 10  solopeprohor                35788 non-null  object        
 11  estfiscod                   43514 non-null  object    

In [9]:
b=base2['soloperiequiflg']
b

0        None
1        None
2        None
3        None
4        None
         ... 
65078    None
65079    None
65080    None
65081    None
65082    None
Name: soloperiequiflg, Length: 65083, dtype: object

In [10]:
b.unique()

array([None], dtype=object)

In [11]:
#CREAMOS LA TABLA TEMPORAL
base2.to_sql(name=f'tmp_{tabla}', con=engine3, if_exists='replace', index=False)

83

In [12]:
#SUBIMOS LA TABLA TEMPORAL AL POSTGRES SQL

query_count_before = f"SELECT COUNT(*) FROM {tabla}"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} antes de la inserción: {cant_antes}")

Cantidad de filas en la tabla qtsop10 antes de la inserción: 1500534


In [13]:
#Borramos en caso el ETL se ejecute una segunda vez
borrando=f"DELETE FROM {tabla} WHERE {col_tabla} >='{fecha}'"
borrado = connection3.execute(borrando)

In [14]:
query=f"""

ALTER TABLE tmp_{tabla} 
ALTER COLUMN solopeoricenasicod TYPE character(1),
ALTER COLUMN solopecenasicod TYPE character(3),
ALTER COLUMN solopenum TYPE numeric(10,0),
ALTER COLUMN solopefec TYPE date,
ALTER COLUMN solopeactmednum TYPE numeric(10,0),
ALTER COLUMN solopemedtipdocidenpercod TYPE character(1),
ALTER COLUMN solopemedperasisdocidennum TYPE character(10),
ALTER COLUMN solopesolfec TYPE date,
ALTER COLUMN solopeprofec TYPE date,
ALTER COLUMN solopeprohor TYPE timestamp,
ALTER COLUMN estfiscod TYPE character(1),
ALTER COLUMN estsopcod TYPE character(1),
ALTER COLUMN solopeestregcod TYPE character(1),
ALTER COLUMN solopeusucrecod TYPE character(10),
ALTER COLUMN solopecrefec TYPE timestamp,
ALTER COLUMN solopeusumodcod TYPE character(10),
ALTER COLUMN solopemodfec TYPE timestamp,
ALTER COLUMN solopecenquicod TYPE character(2),
ALTER COLUMN solopesalopecod TYPE character(2),
ALTER COLUMN solopeesttpo TYPE timestamp,
ALTER COLUMN solopearehoscod TYPE character(2),
ALTER COLUMN solopeservhoscod TYPE character(3),
ALTER COLUMN solopeordnum TYPE numeric(2,0),
ALTER COLUMN solopeemecod TYPE character(2),
ALTER COLUMN solopesolarehoscod TYPE character(2),
ALTER COLUMN solopesolservhoscod TYPE character(3),
ALTER COLUMN conopecod TYPE character(1),
ALTER COLUMN solopeactmedopenum TYPE numeric(10,0),
ALTER COLUMN priopecod TYPE character(1),
ALTER COLUMN riequicod TYPE character(1),
ALTER COLUMN solopediashosprecan TYPE numeric(2,0),
ALTER COLUMN solopediashosposcan TYPE numeric(2,0),
ALTER COLUMN motsopcod TYPE character(2),
ALTER COLUMN solopetipanecod TYPE character(1),
ALTER COLUMN soloperesexalabflg TYPE character(1),
ALTER COLUMN soloperiequiflg TYPE character(1),
ALTER COLUMN soloperieneuflg TYPE character(1),
ALTER COLUMN solopeconinfflg TYPE character(1),
ALTER COLUMN solopeordtraflg TYPE character(1),
ALTER COLUMN solopeevalpqxflg TYPE character(1),
ALTER COLUMN solopeevalpqxfec TYPE date,
ALTER COLUMN solopeevalpqxoricenasicod TYPE character(1),
ALTER COLUMN solopeevalpqxcenasicod TYPE character(3),
ALTER COLUMN solopeevalpqxactmednum TYPE numeric(10,0),
ALTER COLUMN solopetopemecod TYPE character(2),
ALTER COLUMN solopeatesecnum TYPE numeric(4,0),
ALTER COLUMN solopebuspacsecnum TYPE numeric(10,0),
ALTER COLUMN solopesolexafec TYPE date USING solopesolexafec::date,
ALTER COLUMN soloperesexalabfec TYPE date USING soloperesexalabfec::date,
ALTER COLUMN soloperiequifec TYPE date USING soloperiequifec::date,
ALTER COLUMN soloperieneufec TYPE date USING soloperieneufec::date,
ALTER COLUMN solopeevalpqxmedtipdoc TYPE character(1),
ALTER COLUMN solopeevalpqxmeddocnum TYPE character(15),
ALTER COLUMN solopediashospreflg TYPE character(1),
ALTER COLUMN solopediashosposflg TYPE character(1),
ALTER COLUMN solopereqprotflg TYPE character(1);


INSERT INTO {tabla} 
(solopeoricenasicod,solopecenasicod,solopenum,solopefec,solopeactmednum,solopemedtipdocidenpercod,solopemedperasisdocidennum,solopesolfec,solopeprofec,solopeprohor,estfiscod,estsopcod,solopeestregcod,solopeusucrecod,solopecrefec,solopeusumodcod,solopemodfec,solopecenquicod,solopesalopecod,solopeesttpo,solopearehoscod,solopeservhoscod,solopeordnum,solopeemecod,solopesolarehoscod,solopesolservhoscod,conopecod,solopeactmedopenum,priopecod,riequicod,solopediashosprecan,solopediashosposcan,motsopcod,solopetipanecod,soloperesexalabflg,soloperiequiflg,soloperieneuflg,solopeconinfflg,solopeordtraflg,solopeevalpqxflg,solopeevalpqxfec,solopeevalpqxoricenasicod,solopeevalpqxcenasicod,solopeevalpqxactmednum,solopetopemecod,solopeatesecnum,solopebuspacsecnum,solopesolexafec,soloperesexalabfec,soloperiequifec,soloperieneufec,solopeevalpqxmedtipdoc,solopeevalpqxmeddocnum,solopediashospreflg,solopediashosposflg,solopereqprotflg) 

SELECT 
solopeoricenasicod,solopecenasicod,solopenum,solopefec,solopeactmednum,solopemedtipdocidenpercod,solopemedperasisdocidennum,solopesolfec,solopeprofec,solopeprohor,estfiscod,estsopcod,solopeestregcod,solopeusucrecod,solopecrefec,solopeusumodcod,solopemodfec,solopecenquicod,solopesalopecod,solopeesttpo,solopearehoscod,solopeservhoscod,solopeordnum,solopeemecod,solopesolarehoscod,solopesolservhoscod,conopecod,solopeactmedopenum,priopecod,riequicod,solopediashosprecan,solopediashosposcan,motsopcod,solopetipanecod,soloperesexalabflg,soloperiequiflg,soloperieneuflg,solopeconinfflg,solopeordtraflg,solopeevalpqxflg,solopeevalpqxfec,solopeevalpqxoricenasicod,solopeevalpqxcenasicod,solopeevalpqxactmednum,solopetopemecod,solopeatesecnum,solopebuspacsecnum,solopesolexafec,soloperesexalabfec,soloperiequifec,soloperieneufec,solopeevalpqxmedtipdoc,solopeevalpqxmeddocnum,solopediashospreflg,solopediashosposflg,solopereqprotflg




FROM tmp_{tabla};
"""

c1= text(query)
connection3.execute(c1)

In [15]:
query_count_after = f"SELECT COUNT(*) FROM {tabla}"
cant_despues = connection3.execute(query_count_after).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

Cantidad de filas en la tabla qtsop10 después de la inserción: 1500812
La cantidad de filas insertadas fue de: 278


In [16]:
#BORRAMOS LAS TABLAS
query2=f"""
DROP TABLE tmp_{tabla};
"""
c2= text(query2)
cursor=connection3.execute(c2)

In [17]:
connection1.close()
connection2.close()
connection3.close()

In [18]:
engine1.dispose()
engine2.dispose()
engine3.dispose()

AYUDA PARA EXTRAER COLUMNAS Y ESTRUCTURA DE TABLAS (NO ES PARTE DEL ETL)

In [19]:
import re

cadena = """
    solopeoricenasicod character(1) COLLATE pg_catalog."default",
    solopecenasicod character(3) COLLATE pg_catalog."default",
    solopenum numeric(10,0),
    solopefec date,
    solopeactmednum numeric(10,0),
    solopemedtipdocidenpercod character(1) COLLATE pg_catalog."default",
    solopemedperasisdocidennum character(10) COLLATE pg_catalog."default",
    solopesolfec date,
    solopeprofec date,
    solopeprohor timestamp with time zone,
    estfiscod character(1) COLLATE pg_catalog."default",
    estsopcod character(1) COLLATE pg_catalog."default",
    solopeestregcod character(1) COLLATE pg_catalog."default",
    solopeusucrecod character(10) COLLATE pg_catalog."default",
    solopecrefec timestamp with time zone,
    solopeusumodcod character(10) COLLATE pg_catalog."default",
    solopemodfec timestamp with time zone,
    solopecenquicod character(2) COLLATE pg_catalog."default",
    solopesalopecod character(2) COLLATE pg_catalog."default",
    solopeesttpo timestamp with time zone,
    solopearehoscod character(2) COLLATE pg_catalog."default",
    solopeservhoscod character(3) COLLATE pg_catalog."default",
    solopeordnum numeric(2,0),
    solopeemecod character(2) COLLATE pg_catalog."default",
    solopesolarehoscod character(2) COLLATE pg_catalog."default",
    solopesolservhoscod character(3) COLLATE pg_catalog."default",
    conopecod character(1) COLLATE pg_catalog."default",
    solopeactmedopenum numeric(10,0),
    priopecod character(1) COLLATE pg_catalog."default",
    riequicod character(1) COLLATE pg_catalog."default",
    solopediashosprecan numeric(2,0),
    solopediashosposcan numeric(2,0),
    motsopcod character(2) COLLATE pg_catalog."default",
    solopetipanecod character(1) COLLATE pg_catalog."default",
    soloperesexalabflg character(1) COLLATE pg_catalog."default",
    soloperiequiflg character(1) COLLATE pg_catalog."default",
    soloperieneuflg character(1) COLLATE pg_catalog."default",
    solopeconinfflg character(1) COLLATE pg_catalog."default",
    solopeordtraflg character(1) COLLATE pg_catalog."default",
    solopeevalpqxflg character(1) COLLATE pg_catalog."default",
    solopeevalpqxfec date,
    solopeevalpqxoricenasicod character(1) COLLATE pg_catalog."default",
    solopeevalpqxcenasicod character(3) COLLATE pg_catalog."default",
    solopeevalpqxactmednum numeric(10,0),
    solopetopemecod character(2) COLLATE pg_catalog."default",
    solopeatesecnum numeric(4,0),
    solopebuspacsecnum numeric(10,0),
    solopesolexafec date,
    soloperesexalabfec date,
    soloperiequifec date,
    soloperieneufec date,
    solopeevalpqxmedtipdoc character(1) COLLATE pg_catalog."default",
    solopeevalpqxmeddocnum character(15) COLLATE pg_catalog."default",
    solopediashospreflg character(1) COLLATE pg_catalog."default",
    solopediashosposflg character(1) COLLATE pg_catalog."default",
    solopereqprotflg character(1) COLLATE pg_catalog."default"
"""

# Reemplaza los caracteres innecesarios
cadena = cadena.replace(" COLLATE pg_catalog.\"default\",", "")
cadena = cadena.replace(" with time zone", "")

# Divide la cadena en una lista de líneas
lineas = cadena.split("\n")

# Crea la cadena de alteración de columnas
cadena_alter = ""
for i, linea in enumerate(lineas):
    palabras = linea.split()
    if len(palabras) >= 2:
        columna = palabras[0]
        tipo = palabras[1]
        if i == len(lineas) - 2:
            # Última línea, agrega punto y coma
            cadena_alter += f"ALTER COLUMN {columna} TYPE {tipo};\n"
        else:
            # Otras líneas, agrega coma
            cadena_alter += f"ALTER COLUMN {columna} TYPE {tipo},\n"

# Utiliza una expresión regular para eliminar las comas duplicadas
cadena_alter = re.sub(r',+$', ',', cadena_alter, flags=re.MULTILINE)

print(cadena_alter)



import re

nombrecitos = re.findall(r'ALTER COLUMN\s+(\S+)', cadena_alter)
insertado_col = ",".join(nombrecitos)

print(insertado_col)

ALTER COLUMN solopeoricenasicod TYPE character(1),
ALTER COLUMN solopecenasicod TYPE character(3),
ALTER COLUMN solopenum TYPE numeric(10,0),
ALTER COLUMN solopefec TYPE date,
ALTER COLUMN solopeactmednum TYPE numeric(10,0),
ALTER COLUMN solopemedtipdocidenpercod TYPE character(1),
ALTER COLUMN solopemedperasisdocidennum TYPE character(10),
ALTER COLUMN solopesolfec TYPE date,
ALTER COLUMN solopeprofec TYPE date,
ALTER COLUMN solopeprohor TYPE timestamp,
ALTER COLUMN estfiscod TYPE character(1),
ALTER COLUMN estsopcod TYPE character(1),
ALTER COLUMN solopeestregcod TYPE character(1),
ALTER COLUMN solopeusucrecod TYPE character(10),
ALTER COLUMN solopecrefec TYPE timestamp,
ALTER COLUMN solopeusumodcod TYPE character(10),
ALTER COLUMN solopemodfec TYPE timestamp,
ALTER COLUMN solopecenquicod TYPE character(2),
ALTER COLUMN solopesalopecod TYPE character(2),
ALTER COLUMN solopeesttpo TYPE timestamp,
ALTER COLUMN solopearehoscod TYPE character(2),
ALTER COLUMN solopeservhoscod TYPE charac